In [17]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
llm=init_chat_model("groq:qwen/qwen3-32b")
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002AE0B2AD6E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002AE0B2AE060>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage



agent=create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)

In [3]:
config={"configurable":{"thread_id":"test_1"}}

In [4]:
question=[
    "what is 2+2?",
    "what is 1-543?",
    "what is 111*322?",
    "what is 53/134?",
    "what is 23%34?",
    "what is 42-322?",
    "what is 2+2?",
    "what is 254+4434?",
    "what is 543-4324?",
]
for q in question:
    response=agent.invoke({"messages":[HumanMessage(content=q)]}, config)
    print(f"Messages:{response}")
    print(f"Messages: {len(response['messages'])}")

Messages:{'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='fb9a6d0e-dff5-412e-a2f8-b5a094f71b5d'), AIMessage(content='<think>\nOkay, so the user is asking "what is 2+2?" That seems pretty straightforward, but let me think about how to approach this. First, I need to make sure I understand the question correctly. It\'s a basic arithmetic problem, adding two numbers together. The answer is likely 4, but maybe there\'s a trick or a different context here.\n\nWait, could this be a trick question? Sometimes people ask this to see if someone knows if there are alternative interpretations. For example, in some contexts, like in a different base system, 2+2 might not be 4. Let\'s consider that. If we\'re in base 10, which is standard, then 2+2 is definitely 4. But if it\'s in base 3, for instance, 2+2 would be 11 (since 2+2=4 in decimal, which is 1*3^1 + 1*3^0 = 11 in base 3). However, unless specified, the default assumption is base 10.\n\nAnot

KeyboardInterrupt: 

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.tools import tool

@tool
def search_hotels(city:str)->str:
    """Search hotels in a city and return the names of the hotels."""
    return f"""Hotels in {city}:
    1. Grand Hotel
    2. City Inn 
    3. Ocean View Resort""" 

agent=create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens",550),
            keep=("tokens",200)
        )
    ]
)
config={"configurable":{"thread_id":"test_2"}}
def count_tokens(messages):
    return sum(len(m.content.split()) for m in messages)


In [6]:
cities=["Paris","New York","Tokyo","Sydney","Dubai"]
for city in cities:
    response=agent.invoke(
        {"messages" :[HumanMessage(content=f'search hotels in {city}')] },
        config=config
    )

    tokens=count_tokens(response["messages"])
    print(f"Tokens: {tokens}")
    print(f"Messages: {response['messages']}")

Tokens: 46
Messages: [HumanMessage(content='search hotels in Paris', additional_kwargs={}, response_metadata={}, id='770cfdae-3a3f-4d12-8ed2-fbdf7d925af4'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to search for hotels in Paris. Let me check the available tools. There\'s a function called search_hotels that takes a city parameter. The required parameter is city, and it\'s a string. Since the user specified Paris, I need to call this function with "Paris" as the city argument. I\'ll make sure to format the tool call correctly in JSON inside the XML tags.\n', 'tool_calls': [{'id': 'nnh1wz0e0', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 108, 'prompt_tokens': 157, 'total_tokens': 265, 'completion_time': 0.163546789, 'completion_tokens_details': {'reasoning_tokens': 83}, 'prompt_time': 0.007439028, 'prompt_tokens_details': None, 'queue_time':

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3-32b` in organization `org_01kkgsaxx8e0mbymrmeprwacrg` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 4306, Requested 2691. Please try again in 9.969999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [27]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

In [25]:
def read_email_tool(email:str)-> str:
    """Mock function to read an email by its ID"""
    return f"Emaik content for ID: {email}"

def send_email_tool(recipient: str, subject:str, body:str)->str:
    "Mock function to send an email"
    return f"Email sent to {recipient} with subject '{subject}'"

In [34]:
agent = create_agent(
    model=llm,
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {"allowed_decision": ["approve", "edit", "reject"]},
                "read_email_tool": False,
            }
        )
    ]
)


In [48]:
config={"configurable": {"thread_id":"test_approve"}}

result=agent.invoke({"messages":[HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
                    config=config,
                    version="v2")

In [49]:
result

GraphOutput(value={'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='d90643d4-ec09-465b-a13a-2ce4c311fd36'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. All three are required. The parameters are provided in the user's request. So I need to call the send_email_tool with those exact parameters. No need to use the read_email_tool here since the user isn't asking to read an email. Just confirm that all required fields are present. Yep, recipient is john@test.com, subject is Hello, body is How are you?. All set. I'll generate the tool call with those arguments.\n", 'tool_calls': [{'id': 'k82y2w6am', 'function': {'arguments': '{"body":"How are you?","recipient"

In [52]:
if "__interrupt__" in result:
    print("Paused! Approving")
else:
    print("it does not include interrupt")   


it does not include interrupt


C:\Users\hp\AppData\Local\Temp\ipykernel_3716\3009569628.py:1: LangGraphDeprecatedSinceV11: Accessing GraphOutput via `key in result` is deprecated. Use `result.value` to access the output value directly, or `result.interrupts` for interrupts. Deprecated in LangGraph V1.1 to be removed in V3.0.
  if "__interrupt__" in result:
